In [1]:
import ast
import concurrent.futures
import glob
import itertools
import os
import pickle
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

import dask
import dask.dataframe as dd
import dask_ml.cluster as dask_cluster
from dask.distributed import Client
from dask.diagnostics import ProgressBar

from concurrent.futures import ThreadPoolExecutor
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from sklearn.linear_model import LinearRegression
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.cluster import KMeans

from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm
from collections import Counter
from functools import reduce
from pprint import pprint

pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Read all files

In [2]:
def read_csv_helper(fpath):
    splits = fpath.split("=")
    cutoff = splits[2].split(".")[0]
    fips = splits[1].split("_")[0]
    df = pd.read_csv(fpath, header=None)
    
    n_trees = len(df.columns)
    
    df["fips"] = int(fips)
    df["cutoff"] = int(cutoff)
    
    df = df[["fips","cutoff"] + [i for i in range(n_trees)]]
    
    return df

In [ ]:
combined_df_path = "./depths_by_fips_cutoff.csv"
if os.path.exists(combined_df_path):
    print("{} exists! Loading!".format(combined_df_path))
    combined_df = pd.read_csv(combined_df_path)
else:
    print("{} does not exist! Creating!".format(combined_df_path))
    depths_subfolder = "./depths"
    file_paths = [os.path.join(depths_subfolder, filename) for filename in os.listdir(depths_subfolder)]
    with concurrent.futures.ProcessPoolExecutor(max_workers=cpu_count()) as executor:
        # Submit file reading tasks to the executor
        futures = [executor.submit(read_csv_helper, file_path) for file_path in file_paths]

        # Use tqdm to display progress
        progress_bar = tqdm(total=len(futures))
        dfs = []

        # Retrieve the results as they complete
        for future in concurrent.futures.as_completed(futures):
            df = future.result()
            dfs.append(df)
            progress_bar.update(1)

    # Close the progress bar
    progress_bar.close()

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df = combined_df.sort_values(by=["cutoff","fips"])
    print("Writing {}".format(combined_df_path))
    combined_df.to_csv(combined_df_path, index=False)

./depths_by_fips_cutoff.csv does not exist! Creating!


In [ ]:
combined_df

### Depths by Date?

In [ ]:
cutoff_to_date = pd.read_csv("../../cutoff_to_date/cutoff_to_date.csv")
cutoff_to_date["date"] = pd.to_datetime(cutoff_to_date["date"])

In [ ]:
date_combined_df = pd.merge(combined_df, cutoff_to_date, left_on="cutoff", right_on="days_from_start")
date_combined_df.set_index('date', inplace=True)


In [ ]:
columns_to_exclude = ["fips", "cutoff", "days_from_start"]
mean_depths_by_cutoff = date_combined_df.groupby(date_combined_df.index).apply(lambda x : np.mean(x.loc[:, ~x.columns.isin(columns_to_exclude)].values))
median_depths_by_cutoff = date_combined_df.groupby(date_combined_df.index).apply(lambda x : np.median(x.loc[:, ~x.columns.isin(columns_to_exclude)].values))
max_depths_by_cutoff = date_combined_df.groupby(date_combined_df.index).apply(lambda x : np.max(x.loc[:, ~x.columns.isin(columns_to_exclude)].values))
min_depths_by_cutoff = date_combined_df.groupby(date_combined_df.index).apply(lambda x : np.min(x.loc[:, ~x.columns.isin(columns_to_exclude)].values))



In [ ]:
plt.figure(figsize=(20,10))
#plt.plot(max_depths_by_cutoff,label="Max Tree Depth")
plt.plot(mean_depths_by_cutoff,label="Mean Tree Depth")
plt.plot(median_depths_by_cutoff,label="Median Tree Depth")
#plt.plot(min_depths_by_cutoff,label="Min Tree Depth")

plt.ylabel("Tree Depth")
plt.xlabel("Cutoff")
plt.title("Tree Depth of TLGRF Time Only")
plt.xlim(pd.to_datetime("2020-03-15"), pd.to_datetime("2023-01-01"))
plt.ylim(0,4.5)
plt.legend()
plt.show()

In [ ]:
TLGRF_Time_Only_depth_by_date = pd.DataFrame()
TLGRF_Time_Only_depth_by_date["Mean"] = mean_depths_by_cutoff
TLGRF_Time_Only_depth_by_date["Median"] = median_depths_by_cutoff


TLGRF_Time_Only_depth_by_date.to_csv("TLGRF_Time_Only_depth_by_date.csv", index=True)